# Setup

In [ ]:
import duckdb
# import pyarrow
import json
from pathlib import Path

In [ ]:
DATA_DIR = Path('../data/')
RAW_DATA_DIR = DATA_DIR/ 'raw'
WTAT_DUCKDB_FILE = DATA_DIR / 'warehouse' / 'wtat.duckdb'

FACEBOOK_PAGE_FOLDERS = [
    "100064668222513",
    "61569565962838",
    "100082876987694"
]

In [ ]:
fb_page = RAW_DATA_DIR / FACEBOOK_PAGE_FOLDERS[1]

json_posts = [str(j) for j in fb_page.glob("*.json") if j.name != "state.json"]

duckdb.sql(f"""
SELECT regexp_extract(filename, '([^/]+)/[^/]+$', 1) AS page_id, *
FROM read_json_auto(
    {json_posts},
    filename = true,
    union_by_name = true,
    sample_size = -1
)
-- WHERE top_comments IS NULL
ORDER BY timestamp ASC
""").df()

# Move data into duckdb

In [ ]:
DROP_BRONZE_SCHEMA = """
DROP SCHEMA IF EXISTS bronze CASCADE
"""

CREATE_BRONZE_SCHEMA = """
CREATE SCHEMA IF NOT EXISTS bronze
"""

WTAT_DUCKDB_FILE.parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect(WTAT_DUCKDB_FILE) as con:
    con.execute(DROP_BRONZE_SCHEMA)
    con.execute(CREATE_BRONZE_SCHEMA)

all_json_posts = [
    str(j) 
    for fb_page in FACEBOOK_PAGE_FOLDERS
    for j in (RAW_DATA_DIR / fb_page).glob("*.json")
    if j.name != "state.json"
]

with duckdb.connect(WTAT_DUCKDB_FILE) as con: 
    con.sql(f"""
        CREATE OR REPLACE TABLE bronze.posts AS
            SELECT 
                * EXCLUDE(filename),
                filename AS source_file,
                regexp_extract(filename, '([^/]+)/[^/]+$', 1) AS page_id, 
                current_timestamp AS loaded_at
            FROM read_json_auto(
                {all_json_posts},
                filename = true,
                
                sample_size = -1
            );
    """)


In [ ]:
with duckdb.connect(WTAT_DUCKDB_FILE) as con:
    con.sql("SELECT * FROM bronze.posts LIMIT 3").df()